# 🚀 SAFB/ANB 联邦学习后门攻击实验

本项目实现了 **Adaptive Nebula Backdoor (ANB)** 攻击方法。

**仓库地址**: https://github.com/zyh-hnu/ANB

---

## 1️⃣ 环境设置

In [ ]:
# 检查 GPU
import torch

print("="*60)
print("环境信息")
print("="*60)
print(f"PyTorch 版本: {torch.__version__}")
print(f"CUDA 可用: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU 设备: {torch.cuda.get_device_name(0)}")
    print(f"GPU 内存: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️ 警告: 未检测到 GPU，实验将运行缓慢")
print("="*60)

In [ ]:
# 安装依赖
!pip install -q torch torchvision numpy scipy scikit-learn hdbscan matplotlib seaborn tqdm pyyaml Pillow scikit-image opencv-python-headless lpips

## 2️⃣ 克隆项目

In [ ]:
import os
import sys
from pathlib import Path

# 项目目录
PROJECT_DIR = "/content/ANB"

# 克隆仓库
if not os.path.exists(PROJECT_DIR):
    print("正在尝试克隆项目...")
    result = !git clone https://github.com/zyh-hnu/ANB.git {PROJECT_DIR}
    if not os.path.exists(PROJECT_DIR):
        print("❌ 克隆失败。请检查网络或仓库权限。")
        print("💡 建议：您可以手动在左侧文件栏上传项目文件夹，并命名为 'ANB'")
    else:
        print("✅ 克隆成功")
else:
    print("项目已存在，拉取最新代码...")
    !cd {PROJECT_DIR} && git pull

# 只有在目录存在时才切换
if os.path.exists(PROJECT_DIR):
    os.chdir(PROJECT_DIR)
    if PROJECT_DIR not in sys.path:
        sys.path.insert(0, PROJECT_DIR)
    print(f"\n✓ 当前工作目录: {os.getcwd()}")
else:
    print("\n⚠️ 未能进入项目目录，请确保目录存在。")

In [ ]:
# 验证项目结构
!ls -la

## 3️⃣ 验证模块导入

In [ ]:
print("验证模块导入...")

try:
    from config import Config, load_config
    print("  ✓ config")
    
    from core.attacks import AdaptiveNebulaBackdoor, FrequencyBackdoor
    print("  ✓ core.attacks")
    
    from core.defenses import aggregate_with_defense
    print("  ✓ core.defenses")
    
    from federated.client import Client, create_clients
    print("  ✓ federated.client")
    
    from federated.server import Server, federated_training
    print("  ✓ federated.server")
    
    from models.resnet import resnet18
    print("  ✓ models.resnet")
    
    print("\n✅ 所有模块导入成功!")
    
except ImportError as e:
    print(f"\n❌ 导入失败: {e}")
    raise

---
## 4️⃣ 实验配置

**修改下方参数后运行此单元格**

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 实验配置 - 修改此处参数
# ═══════════════════════════════════════════════════════════════

CONFIG = {
    # --- 攻击参数 ---
    "poison_rate": 0.2,           # 恶意客户端毒化比例 (0~1)
    "scaling_factor": 4.0,         # 模型替换放大因子
    "backdoor_boost_weight": 0.1,  # 后门增强损失权重
    "epsilon": 0.1,                # 触发器注入强度
    "target_label": 0,             # 后门目标类
    "freq_strategy": "ANB",        # 攻击策略: "ANB" 或 "FIXED"
    
    # --- 训练参数 ---
    "num_rounds": 10,              # 联邦学习轮数
    "local_epochs": 3,             # 本地训练轮数
    "learning_rate": 0.01,         # 学习率
    "batch_size": 32,              # 批次大小
    "seed": 42,                    # 随机种子
    
    # --- 防御参数 ---
    "defense_enabled": True,       # 是否启用防御
    "defense_method": "hdbscan",   # 防御方法: hdbscan, kmeans, dbscan
    
    # --- 其他 ---
    "num_clients": 10,             # 客户端数量
    
    # --- 消融实验开关 ---
    "use_phased_chaos": True,      # 动态相位调度
    "use_spectral_smoothing": True, # 高斯扩散
    "use_freq_sharding": True,     # 频率分片
    "use_dual_routing": True,      # 双域路由
}

# 打印配置
print("="*60)
print("实验配置")
print("="*60)
for key, value in CONFIG.items():
    print(f"  {key}: {value}")
print("="*60)

---
## 5️⃣ 运行实验

In [ ]:
# 运行实验
cmd_args = " ".join([
    f"--poison-rate {CONFIG['poison_rate']}",
    f"--scaling-factor {CONFIG['scaling_factor']}",
    f"--backdoor-boost-weight {CONFIG['backdoor_boost_weight']}",
    f"--epsilon {CONFIG['epsilon']}",
    f"--target-label {CONFIG['target_label']}",
    f"--freq-strategy {CONFIG['freq_strategy']}",
    f"--num-rounds {CONFIG['num_rounds']}",
    f"--local-epochs {CONFIG['local_epochs']}",
    f"--learning-rate {CONFIG['learning_rate']}",
    f"--batch-size {CONFIG['batch_size']}",
    f"--seed {CONFIG['seed']}",
    f"--num-clients {CONFIG['num_clients']}",
    f"--defense-enabled {1 if CONFIG['defense_enabled'] else 0}",
    f"--defense-method {CONFIG['defense_method']}",
    f"--use-phased-chaos {1 if CONFIG['use_phased_chaos'] else 0}",
    f"--use-spectral-smoothing {1 if CONFIG['use_spectral_smoothing'] else 0}",
    f"--use-freq-sharding {1 if CONFIG['use_freq_sharding'] else 0}",
    f"--use-dual-routing {1 if CONFIG['use_dual_routing'] else 0}",
])

print("执行命令:")
print(f"python main.py {cmd_args}")
print("\n" + "="*60)
print("开始训练...")
print("="*60 + "\n")

!python main.py {cmd_args}

---
## 6️⃣ 查看结果

In [ ]:
import json
import os
from pathlib import Path

# 查找结果文件
results_dir = Path('results')
result_files = list(results_dir.glob('history_*.json')) if results_dir.exists() else []

if result_files:
    # 使用最新的结果文件
    result_file = sorted(result_files, key=lambda x: x.stat().st_mtime)[-1]
    
    with open(result_file, 'r') as f:
        history = json.load(f)
    
    print("="*60)
    print("实验结果")
    print("="*60)
    print(f"结果文件: {result_file.name}")
    print(f"训练轮数: {len(history['test_acc'])}")
    print("-"*60)
    print(f"最终 ACC (Clean Accuracy):  {history['test_acc'][-1]:.2%}")
    print(f"最终 ASR (Single Trigger):  {history['test_asr'][-1]:.2%}")
    
    if history.get('test_asr_multi'):
        print(f"最终 ASR (Multi-Trigger):   {history['test_asr_multi'][-1]:.2%}")
    
    if history.get('defense_bypass_rate'):
        print(f"最终 Bypass Rate:           {history['defense_bypass_rate'][-1]:.2%}")
    
    print("-"*60)
    
    # 判断是否达标
    acc_ok = history['test_acc'][-1] >= 0.60
    asr_ok = history['test_asr'][-1] >= 0.85
    
    print("达标情况:")
    print(f"  ACC ≥ 60%:     {'✓ 达标' if acc_ok else '✗ 未达标'} ({history['test_acc'][-1]:.2%})")
    print(f"  ASR ≥ 85%:     {'✓ 达标' if asr_ok else '✗ 未达标'} ({history['test_asr'][-1]:.2%})")
    
    print("="*60)
else:
    print("⚠️ 未找到结果文件")
    print(f"结果目录: {results_dir.absolute()}")

In [ ]:
# 可视化结果
import matplotlib.pyplot as plt
import numpy as np

if 'history' in dir():
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    
    rounds = range(1, len(history['test_acc']) + 1)
    
    # ACC 曲线
    axes[0, 0].plot(rounds, [x*100 for x in history['test_acc']], 'b-', linewidth=2, marker='o', markersize=4)
    axes[0, 0].axhline(y=60, color='g', linestyle='--', label='Target (60%)')
    axes[0, 0].set_xlabel('Round')
    axes[0, 0].set_ylabel('Accuracy (%)')
    axes[0, 0].set_title('Clean Accuracy')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    axes[0, 0].set_ylim([0, 100])
    
    # ASR 曲线
    axes[0, 1].plot(rounds, [x*100 for x in history['test_asr']], 'r-', linewidth=2, marker='o', markersize=4, label='Single')
    if history.get('test_asr_multi'):
        axes[0, 1].plot(rounds, [x*100 for x in history['test_asr_multi']], 'r--', linewidth=2, marker='s', markersize=4, label='Multi')
    axes[0, 1].axhline(y=85, color='g', linestyle='--', label='Target (85%)')
    axes[0, 1].set_xlabel('Round')
    axes[0, 1].set_ylabel('ASR (%)')
    axes[0, 1].set_title('Attack Success Rate')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)
    axes[0, 1].set_ylim([0, 105])
    
    # Bypass Rate 曲线
    if history.get('defense_bypass_rate'):
        axes[1, 0].plot(rounds, [x*100 for x in history['defense_bypass_rate']], 'g-', linewidth=2, marker='o', markersize=4)
        axes[1, 0].axhline(y=70, color='orange', linestyle='--', label='Target (70%)')
        axes[1, 0].set_xlabel('Round')
        axes[1, 0].set_ylabel('Bypass Rate (%)')
        axes[1, 0].set_title('Defense Bypass Rate')
        axes[1, 0].legend()
        axes[1, 0].grid(True, alpha=0.3)
        axes[1, 0].set_ylim([0, 105])
    
    # Training Loss 曲线
    if history.get('train_loss'):
        axes[1, 1].plot(rounds, history['train_loss'], 'purple', linewidth=2, marker='o', markersize=4)
        axes[1, 1].set_xlabel('Round')
        axes[1, 1].set_ylabel('Loss')
        axes[1, 1].set_title('Training Loss')
        axes[1, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('results/experiment_results.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("\n✓ 图表已保存到 results/experiment_results.png")
else:
    print("⚠️ 请先运行实验")

---
## 7️⃣ 查看生成的可视化图表

In [ ]:
# 列出生成的图表
import glob
from IPython.display import Image, display

png_files = sorted(glob.glob('results/*.png'))

if png_files:
    print("生成的可视化图表:")
    for f in png_files:
        print(f"  - {f}")
    
    # 显示部分图表
    print("\n显示部分图表:")
    for img_path in png_files[:4]:
        print(f"\n{img_path}:")
        display(Image(filename=img_path, width=500))
else:
    print("⚠️ 未找到可视化图表")

---
## 8️⃣ 下载结果（可选）

In [ ]:
# 打包结果文件以便下载
from google.colab import files

# 压缩结果目录
!zip -r results.zip results/

# 下载
files.download('results.zip')

---
## 📋 参数说明

| 参数 | 默认值 | 说明 |
|------|--------|------||
| `poison_rate` | 0.2 | 恶意客户端毒化比例 |
| `scaling_factor` | 4.0 | 模型替换放大因子 |
| `backdoor_boost_weight` | 0.1 | 后门增强损失权重 |
| `num_rounds` | 10 | 联邦学习轮数 |
| `local_epochs` | 3 | 本地训练轮数 |
| `freq_strategy` | ANB | 攻击策略 |

**预期结果**：ACC ≥ 60%, ASR ≥ 85%